In [1]:
# folder with text files
folder_with_text = './text'

# max text length
max_length=512

# Model internal dim
internal_dim=256

# training batch size
batch_size=128

# where to save model checkpoints
checkpoint_path = 'runs/test-autoregressive-attn-alibi'

# number of epochs to train
num_epochs=100

# Create tokenizer

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
from typing import List, Dict
import os
from kemsekov_torch.text_tools import SimpleTokenizer


txt_files = [[os.path.join(fdir,f) for f in files if f.endswith(".txt")] for fdir,_,files in os.walk(folder_with_text)]
txt_files = [b for a in txt_files for b in a]
txt_lines = [open(v).read() for v in txt_files]

tokenizer = SimpleTokenizer(txt_lines,lowercase=True,unknown_symbols_placeholder=' ')
torch.jit.script(tokenizer).save("tokenizer.pt")

test_str="This is my TEST string! Раз!"
inds=tokenizer.encode(test_str)
print(test_str)
print(inds)
print(tokenizer.decode(inds))

Text length analysis
text lines	 79295
line chars mean	 78.267
line chars std	 180.905
0.05 quantile	 0.0
0.95 quantile	 341.0
0.995 quantile	 711.0
This is my TEST string! Раз!
tensor([54, 42, 43, 53,  1, 43, 53,  1, 47, 59,  1, 54, 39, 53, 54,  1, 53, 54,
        52, 43, 48, 41,  2,  1,  1,  1,  1,  2])
this is my test string!    !


/home/bochkarev/Programs/venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


# Define Dataset

In [3]:
import math
from kemsekov_torch.train import split_dataset
import torch
from kemsekov_torch.text_tools import TokenDataset

txt_split = [b for v in txt_lines if len(v)>30 for b in v.split('\n')]
dataset = TokenDataset(
    tokenizer,
    txt_split,
    pad_token=tokenizer.unknown_symbols_placeholder,
    batch_size=batch_size,
    fixed_length=max_length
)

# split dataset into train and test
train_dataset,test_dataset,train_loader, test_loader = split_dataset(
    dataset,
    test_size=0.05,
    batch_size=batch_size,
    random_state=None,
    # bin_by_size=True,
    num_workers=1,
)

Train items 36358
Test items 1914


In [4]:
import random
ind = random.randint(0,len(train_dataset)-1)
inds = dataset[ind]

print("Text length",len(inds))
skip_text = tokenizer.decode(inds).strip()
print(skip_text)

for t in train_loader: break
print("batch size sample",t.shape)

Text length 512
bill's table caught charlie's with a huge bang and knocked one of its legs off. there was a clatter from overhead, and they all looked up to see percy's head poking out of a window on the second floor.


batch size sample torch.Size([128, 512])


# Define Model

In [5]:
from autoregressive import AutoregressiveChar

model = AutoregressiveChar(tokenizer.vocab_size,internal_dim,layers=3,mlp_factor=4,impl='attn')
print(model.params_count())
[c.shape for c in model(t[:,:32])]

3584771


[torch.Size([128, 32, 256]), torch.Size([128, 32, 80])]

# Training

In [ ]:
from kemsekov_torch.train import train
from kemsekov_torch.metrics import f1_score
from accelerate.utils import TorchDynamoPlugin
from torchmetrics.classification import MulticlassF1Score

# Initialize the metric object
f1_metric = MulticlassF1Score(num_classes=tokenizer.vocab_size, average='macro').cuda()

CE = torch.nn.CrossEntropyLoss()


def get_pad_mask(next_t: torch.Tensor, pad_token: int) -> torch.Tensor:
    """
    Finds the first occurrence of 3 sequential pad tokens per batch row 
    and returns a flattened boolean mask of shape [BATCH * seqlen].
    Elements at and after the 3 pads are set to False.
    
    We use this thing to compute loss on non-padded part of batch
    """
    is_pad = (next_t == pad_token)
    
    # Check 3 sequential tokens using slicing
    sequential_3_pads = is_pad[:, :-2] & is_pad[:, 1:-1] & is_pad[:, 2:]
    
    # Find the first index along dim=1 where this happens per batch item
    has_3_pads = sequential_3_pads.any(dim=-1, keepdim=True)
    first_pad_idx = torch.argmax(sequential_3_pads.int(), dim=-1, keepdim=True)
    
    # Create index grid to build the mask
    seq_indices = torch.arange(next_t.shape[1], device=next_t.device).unsqueeze(0)
    
    # Retain elements before the 3-pad boundary
    mask = torch.ones_like(next_t, dtype=torch.bool)
    mask = torch.where(has_3_pads, seq_indices < first_pad_idx, mask)
    
    return mask

def compute_loss_and_metric(model,batch):
    prev_t = batch[:,:-1]
    next_t = batch[:,1:]
    activations,logits = model(prev_t)
    mask = get_pad_mask(next_t, pad_token=dataset.pad_token[0]).flatten()
    
    logits=logits.view(-1,logits.shape[-1])[mask]
    next_t=next_t.flatten()[mask]
    
    loss = CE(logits,next_t)
    f1 = f1_metric(logits,next_t)
    return loss,{
        'f1':f1
    }

_ = train(
    model,
    train_loader,
    test_loader,
    compute_loss_and_metric,
    checkpoint_path,
    # f'{checkpoint_path}/last',
    gradient_clipping_max_norm=1,
    accelerate_args=dict(
        mixed_precision='fp16',
        dynamo_plugin = TorchDynamoPlugin(
            backend="inductor",
            mode="default",
            fullgraph=False,
            dynamic=True
        )
    ),
    save_on_metric_improve=['f1'],
    num_epochs=num_epochs,
    checkpoints_count=1,
    # default_lr=0.01
)

/home/bochkarev/Programs/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using dir runs/test-autoregressive-attn-alibi
Using default fused AdamW optimizer
Using default CosineAnelingScheduler
Total model parameters 3.58 M
Using device cuda

Epoch 1/100


train 0: 100%|██████████| 284/284 [01:00<00:00,  4.71it/s, f1=0.3340, loss=1.3922]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 1.81108 | 1.38059 |
|  f1  | 0.2222  | 0.3299  |
+------+---------+---------+
saved epoch-1

Epoch 2/100


train 0: 100%|██████████| 284/284 [00:31<00:00,  9.05it/s, f1=0.4012, loss=1.2065]


...

train 0: 100%|██████████| 284/284 [00:31<00:00,  8.88it/s, f1=0.5025, loss=0.9577]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 0.96701 | 1.07367 |
|  f1  | 0.4787  | 0.4452  |
+------+---------+---------+

Epoch 11/100


train 0: 100%|██████████| 284/284 [00:31<00:00,  8.90it/s, f1=0.4865, loss=0.9447]


+------+---------+---------+
|      |  Train  |  Test   |
+------+---------+---------+
| loss | 0.95365 | 1.07647 |
|  f1  | 0.4848  | 0.4410  |
+------+---------+---------+

Epoch 12/100


train 0:   0%|          | 0/284 [00:00<?, ?it/s]

In [ ]:
!python sample_gd2.py --prompt "The Hanged Man, the village pub" --to_generate 1024

/home/bochkarev/Programs/AutoregressiveChar/sample_gd2.py:16: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  tokenizer=torch.load("tokenizer.pt",weights_only=False)
/home/bochkarev/Programs/venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
Loading last checkpoint at epoch 10
the hanged man, the village publicity could stared to sitting to sit feeling in the beaten a deluminating as they had a disaberfely back to him and the door. the spellophineas the only saw long, rosmerta when the door. harry points the ministry of the large back on the walked harry had looked at him out of his father his wand back into his owner than he was now was now where their the body had disappeared to silence. he was listened to respon shaking to staring to go back as he